In [0]:
from pyspark.sql import functions as f
import sys
sys.path.append('..')
sys.path.append('../..')

import lib_etl.validations_ETL as validations
from lib.job_manager import load_config, split_config

In [0]:
%run ../../config/utils

In [0]:
config = load_config(etl_config_path)
data_paths, club_square_config, config_validation = split_config(config)
run_as_date = dbutils.widgets.get("run_as_date")
quotient_out = {"MBRSHP_SID", "MBRSHP_NBR", "USERCODE", "SIGNUP"}

### Transform 

In [0]:
# recency_lookback_duration = data_paths.get("recency_lookback_duration", {}) TODO
# etl_input_data_validator(
#     "intermediate",
#     recency_lookback_duration,
#     data_paths,
#     ["quotient_id_extended"],
# )

In [0]:
df = spark.table(bronze_quotient_id)

member_extended = spark.table(silver_master_member_extended)
member_extended = member_extended.select(["MBRSHP_NBR", "MBRSHP_SID"])
df = df.withColumnRenamed("LOYALTYNUMBER", "MBRSHP_NBR").join(
    member_extended, "MBRSHP_NBR", "inner"
)

df_quotient_id = df.select(*quotient_out).withColumn("HAS_QUOTIENT_ID", f.lit(1))

df_quotient_id.createOrReplaceTempView("source")

### Merge

In [0]:
df_quotient_id.write.mode("overwrite").saveAsTable(silver_quotient_id)

if archive_flag:
    save_archive(df_quotient_id, silver_quotient_id_archive, run_as_date)